# 16 — Callbacks and Observability

Monitor LLM calls, track tokens, estimate costs, and trace chain execution.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
import time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult

## Custom Callback Handlers

In [ ]:
class ObservabilityHandler(BaseCallbackHandler):
    def __init__(self):
        self.calls, self.total_tokens, self.total_cost = [], 0, 0.0
        self._start_times = {}

    def on_llm_start(self, serialized, prompts, **kwargs):
        run_id = str(kwargs.get("run_id", "unknown"))
        self._start_times[run_id] = time.time()
        print(f"    [LLM Start] run_id: {run_id[:8]}")

    def on_llm_end(self, response: LLMResult, **kwargs):
        run_id = str(kwargs.get("run_id", "unknown"))
        elapsed = time.time() - self._start_times.pop(run_id, time.time())
        token_usage = response.llm_output.get("token_usage", {}) if response.llm_output else {}
        prompt_tokens = token_usage.get("prompt_tokens", 0)
        completion_tokens = token_usage.get("completion_tokens", 0)
        total = prompt_tokens + completion_tokens
        cost = (prompt_tokens * 0.15 + completion_tokens * 0.6) / 1_000_000
        self.total_tokens += total
        self.total_cost += cost
        self.calls.append({"latency_ms": round(elapsed * 1000), "tokens": total, "cost_usd": cost})
        print(f"    [LLM End] {elapsed:.2f}s | {total} tokens | ${cost:.6f}")

    def summary(self):
        return f"Total calls: {len(self.calls)} | Tokens: {self.total_tokens} | Cost: ${self.total_cost:.6f}"

## Run Observed Chains

In [ ]:
handler = ObservabilityHandler()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, callbacks=[handler])
chain = ChatPromptTemplate.from_template("Explain {topic} in one sentence.") | llm | StrOutputParser()

for topic in ["recursion", "hash tables", "Big O notation"]:
    print(f"Topic: {topic}")
    result = chain.invoke({"topic": topic}, config={"callbacks": [handler]})
    print(f"Result: {result}\n")

print(f"\n{handler.summary()}")